In [1]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.7 MB/s eta 0:00:00


In [ ]:
import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id='AKIA6LA4HRxxxxxxxxYM',
    aws_secret_access_key='hezvoVSKEnSN5tiIW/zxXxxxxxxxxxxxPyz5rOp',
    region_name='ap-south-1'  # change if your bucket region is different
)

# Test connection
response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

bronze-tokyo-olympics-2020
gold-tokyo-olympics-2020
silver-tokyo-olympics-2020


In [15]:
response = s3.list_objects_v2(Bucket='bronze-tokyo-olympics-2020')

for obj in response['Contents']:
    print(obj['Key'])

2020_Olympics_Dataset.csv.zip


In [17]:
import io
import zipfile
import pandas as pd

# Step 1: Get zip file from S3
obj = s3.get_object(
    Bucket='bronze-tokyo-olympics-2020',
    Key='2020_Olympics_Dataset.csv.zip'
)

# Step 2: Read zip content
zip_data = obj['Body'].read()

# Step 3: Open zip file
zip_file = zipfile.ZipFile(io.BytesIO(zip_data))

# Step 4: See what is inside zip
print(zip_file.namelist())

['athletes.csv', 'coaches.csv', 'medals.csv', 'medals_total.csv', 'technical_officials.csv']


In [18]:
# Automatically take first file inside zip
csv_filename = zip_file.namelist()[0]

# Read CSV into dataframe
df = pd.read_csv(zip_file.open(csv_filename))

# Show first 5 rows
df.head()

,name,short_name,gender,birth_date,birth_place,birth_country,country,country_code,discipline,discipline_code,residence_place,residence_country,height_m/ft,url
0,AALERUD Katrine,AALERUD K,Female,1994-12-04,VESTBY,Norway,Norway,NOR,Cycling Road,CRD,NaN,NaN,NaN,../../../en/results/cycling-road/athlete-profi...
1,ABAD Nestor,ABAD N,Male,1993-03-29,ALCOI,Spain,Spain,ESP,Artistic Gymnastics,GAR,MADRID,Spain,1.65/5'4'',../../../en/results/artistic-gymnastics/athlet...
2,ABAGNALE Giovanni,ABAGNALE G,Male,1995-01-11,GRAGNANO,Italy,Italy,ITA,Rowing,ROW,SABAUDIA,Italy,1.98/6'5'',../../../en/results/rowing/athlete-profile-n13...
3,ABALDE Alberto,ABALDE A,Male,1995-12-15,FERROL,Spain,Spain,ESP,Basketball,BKB,NaN,NaN,2.00/6'6'',../../../en/results/basketball/athlete-profile...
4,ABALDE Tamara,ABALDE T,Female,1989-02-06,VIGO,Spain,Spain,ESP,Basketball,BKB,NaN,NaN,1.92/6'3'',../../../en/results/basketball/athlete-profile...


In [19]:
# Load all files from zip into separate dataframes

athletes_df = pd.read_csv(zip_file.open('athletes.csv'))
coaches_df = pd.read_csv(zip_file.open('coaches.csv'))
medals_df = pd.read_csv(zip_file.open('medals.csv'))
medals_total_df = pd.read_csv(zip_file.open('medals_total.csv'))
technical_officials_df = pd.read_csv(zip_file.open('technical_officials.csv'))

print("All files loaded successfully!")

All files loaded successfully!


In [20]:
athletes_df.head()

,name,short_name,gender,birth_date,birth_place,birth_country,country,country_code,discipline,discipline_code,residence_place,residence_country,height_m/ft,url
0,AALERUD Katrine,AALERUD K,Female,1994-12-04,VESTBY,Norway,Norway,NOR,Cycling Road,CRD,NaN,NaN,NaN,../../../en/results/cycling-road/athlete-profi...
1,ABAD Nestor,ABAD N,Male,1993-03-29,ALCOI,Spain,Spain,ESP,Artistic Gymnastics,GAR,MADRID,Spain,1.65/5'4'',../../../en/results/artistic-gymnastics/athlet...
2,ABAGNALE Giovanni,ABAGNALE G,Male,1995-01-11,GRAGNANO,Italy,Italy,ITA,Rowing,ROW,SABAUDIA,Italy,1.98/6'5'',../../../en/results/rowing/athlete-profile-n13...
3,ABALDE Alberto,ABALDE A,Male,1995-12-15,FERROL,Spain,Spain,ESP,Basketball,BKB,NaN,NaN,2.00/6'6'',../../../en/results/basketball/athlete-profile...
4,ABALDE Tamara,ABALDE T,Female,1989-02-06,VIGO,Spain,Spain,ESP,Basketball,BKB,NaN,NaN,1.92/6'3'',../../../en/results/basketball/athlete-profile...


In [21]:
medals_df.head()

,medal_type,medal_code,medal_date,athlete_short_name,athlete_name,athlete_sex,athlete_link,country_code,discipline_code,event,country,discipline
0,Gold Medal,1,2021-07-24 00:00:00.0,KIM JD,KIM Je Deok,X,../../../en/results/archery/athlete-profile-n1...,KOR,ARC,Mixed Team,Republic of Korea,Archery
1,Gold Medal,1,2021-07-24 00:00:00.0,AN S,AN San,X,../../../en/results/archery/athlete-profile-n1...,KOR,ARC,Mixed Team,Republic of Korea,Archery
2,Silver Medal,2,2021-07-24 00:00:00.0,SCHLOESSER G,SCHLOESSER Gabriela,X,../../../en/results/archery/athlete-profile-n1...,NED,ARC,Mixed Team,Netherlands,Archery
3,Silver Medal,2,2021-07-24 00:00:00.0,WIJLER S,WIJLER Steve,X,../../../en/results/archery/athlete-profile-n1...,NED,ARC,Mixed Team,Netherlands,Archery
4,Bronze Medal,3,2021-07-24 00:00:00.0,ALVAREZ L,ALVAREZ Luis,X,../../../en/results/archery/athlete-profile-n1...,MEX,ARC,Mixed Team,Mexico,Archery


In [22]:
medals_total_df = medals_total_df.drop_duplicates()
medals_total_df.columns = medals_total_df.columns.str.lower()

medals_total_df.head()

,rank,country code,gold medal,silver medal,bronze medal,total,country
0,1,USA,39,41,33,113,United States of America
1,2,CHN,38,32,18,88,People's Republic of China
2,3,JPN,27,14,17,58,Japan
3,4,GBR,22,21,22,65,Great Britain
4,5,ROC,20,28,23,71,ROC


In [23]:
medals_total_df.to_csv("silver_medals_total.csv", index=False)

print("Silver file created successfully")

Silver file created successfully


In [25]:
s3.upload_file(
    "silver_medals_total.csv",
    "silver-tokyo-olympics-2020",
    "silver_medals_total.csv"
)

print("Uploaded to Silver bucket successfully")

Uploaded to Silver bucket successfully


In [26]:
gold_df = medals_total_df.sort_values(by="total", ascending=False).head(10)

gold_df.to_csv("gold_top10_countries.csv", index=False)

gold_df.head()

,rank,country code,gold medal,silver medal,bronze medal,total,country
0,1,USA,39,41,33,113,United States of America
1,2,CHN,38,32,18,88,People's Republic of China
4,5,ROC,20,28,23,71,ROC
3,4,GBR,22,21,22,65,Great Britain
2,3,JPN,27,14,17,58,Japan


In [27]:
s3.upload_file(
    "gold_top10_countries.csv",
    "gold-tokyo-olympics-2020",
    "gold_top10_countries.csv"
)

print("Uploaded to Gold bucket successfully")

Uploaded to Gold bucket successfully
